In [1]:
# =========================
# 1. Mount Google Drive
# =========================
from google.colab import drive
drive.mount('/content/drive')

# =========================
# 2. Define Base Path
# =========================
import os

BASE_PATH = "/content/drive/MyDrive/Plan A"

folders = [
    "data/mimiciv",
    "data/eicu",
    "data/mimiciii",
    "processed",
    "final",
    "results",
    "models",
    "logs"
]

for f in folders:
    os.makedirs(os.path.join(BASE_PATH, f), exist_ok=True)

print("Folder structure ready.")

# =========================
# 3. Install + Import BigQuery
# =========================
!pip install --quiet google-cloud-bigquery

from google.cloud import bigquery

# Authenticate
from google.colab import auth
auth.authenticate_user()

client = bigquery.Client(project="mimic-rs")

print("BigQuery connected.")

# =========================
# 4. Define Query (MIMIC-IV Cohort)
# =========================

query = """
WITH first_icu AS (
  SELECT
    icu.subject_id,
    icu.hadm_id,
    icu.stay_id,
    icu.intime,
    icu.outtime,
    ROW_NUMBER() OVER (PARTITION BY icu.subject_id ORDER BY icu.intime) AS rn
  FROM `physionet-data.mimiciv_3_1_icu.icustays` icu
),

cohort AS (
  SELECT
    f.subject_id,
    f.hadm_id,
    f.stay_id,
    f.intime,
    f.outtime,
    pat.gender,
    EXTRACT(YEAR FROM f.intime) - pat.anchor_year + pat.anchor_age AS age,
    adm.hospital_expire_flag
  FROM first_icu f
  JOIN `physionet-data.mimiciv_3_1_hosp.patients` pat
    ON f.subject_id = pat.subject_id
  JOIN `physionet-data.mimiciv_3_1_hosp.admissions` adm
    ON f.hadm_id = adm.hadm_id
  WHERE f.rn = 1
)

SELECT *
FROM cohort
WHERE age >= 18
"""

# =========================
# 5. Run Query
# =========================
df = client.query(query).to_dataframe()

print("Shape:", df.shape)
df.head()

# =========================
# 6. Save (overwrite-safe)
# =========================
save_path = os.path.join(BASE_PATH, "data/mimiciv/mimiciv_cohort.csv")
df.to_csv(save_path, index=False)

print(f"Saved to: {save_path}")

Mounted at /content/drive
Folder structure ready.
BigQuery connected.
Shape: (65366, 8)
Saved to: /content/drive/MyDrive/Plan A/data/mimiciv/mimiciv_cohort.csv


In [2]:
df['age'].describe()

,age
count,65366.0
mean,64.536579
std,17.127917
min,18.0
25%,54.0
50%,66.0
75%,78.0
max,103.0


In [3]:
df['hospital_expire_flag'].mean()

np.float64(0.10840498118287795)

In [4]:
df['subject_id'].nunique(), df.shape[0]

(65366, 65366)

In [5]:
df.isnull().sum()

,0
subject_id,0
hadm_id,0
stay_id,0
intime,0
outtime,12
gender,0
age,0
hospital_expire_flag,0


In [12]:
# =========================
# Cell 1
# MIMIC-IV: heart rate extraction (first 24h)
# Publication-grade, overwrite-safe
# =========================

import os
import pandas as pd
from google.colab import auth
from google.cloud import bigquery

# Authenticate and connect
auth.authenticate_user()
client = bigquery.Client(project="mimic-rs")

# Base folder
BASE_PATH = "/content/drive/MyDrive/Plan A"
mimiciv_dir = os.path.join(BASE_PATH, "data", "mimiciv")
os.makedirs(mimiciv_dir, exist_ok=True)

# Exact heart rate itemid only
HEART_RATE_ITEMID = 220045

# Build query
hr_query = f"""
WITH first_icu AS (
  SELECT
    icu.subject_id,
    icu.hadm_id,
    icu.stay_id,
    icu.intime,
    icu.outtime,
    ROW_NUMBER() OVER (PARTITION BY icu.subject_id ORDER BY icu.intime) AS rn
  FROM `physionet-data.mimiciv_3_1_icu.icustays` icu
),
cohort AS (
  SELECT
    f.subject_id,
    f.hadm_id,
    f.stay_id,
    f.intime,
    f.outtime,
    pat.gender,
    EXTRACT(YEAR FROM f.intime) - pat.anchor_year + pat.anchor_age AS age,
    adm.hospital_expire_flag
  FROM first_icu f
  JOIN `physionet-data.mimiciv_3_1_hosp.patients` pat
    ON f.subject_id = pat.subject_id
  JOIN `physionet-data.mimiciv_3_1_hosp.admissions` adm
    ON f.hadm_id = adm.hadm_id
  WHERE f.rn = 1
    AND (EXTRACT(YEAR FROM f.intime) - pat.anchor_year + pat.anchor_age) >= 18
),
hr_events AS (
  SELECT
    c.stay_id,
    ce.charttime,
    SAFE_CAST(ce.valuenum AS FLOAT64) AS heart_rate
  FROM cohort c
  JOIN `physionet-data.mimiciv_3_1_icu.chartevents` ce
    ON ce.stay_id = c.stay_id
   AND ce.charttime >= c.intime
   AND ce.charttime < TIMESTAMP_ADD(c.intime, INTERVAL 24 HOUR)
   AND ce.itemid = {HEART_RATE_ITEMID}
  WHERE ce.valuenum IS NOT NULL
    AND SAFE_CAST(ce.valuenum AS FLOAT64) BETWEEN 30 AND 220
)

SELECT
  c.subject_id,
  c.hadm_id,
  c.stay_id,
  c.intime,
  c.outtime,
  c.gender,
  c.age,
  c.hospital_expire_flag,
  COUNT(h.heart_rate) AS hr_n,
  MIN(h.heart_rate) AS hr_min,
  MAX(h.heart_rate) AS hr_max,
  AVG(h.heart_rate) AS hr_mean,
  ARRAY_AGG(h.heart_rate ORDER BY h.charttime LIMIT 1)[OFFSET(0)] AS hr_first,
  ARRAY_AGG(h.heart_rate ORDER BY h.charttime DESC LIMIT 1)[OFFSET(0)] AS hr_last
FROM cohort c
LEFT JOIN hr_events h
  ON c.stay_id = h.stay_id
GROUP BY
  c.subject_id, c.hadm_id, c.stay_id, c.intime, c.outtime,
  c.gender, c.age, c.hospital_expire_flag
ORDER BY c.subject_id
"""

# Run
hr_df = client.query(hr_query).to_dataframe()

print("Shape:", hr_df.shape)
display(hr_df.head())

Shape: (65366, 14)


,subject_id,hadm_id,stay_id,intime,outtime,gender,age,hospital_expire_flag,hr_n,hr_min,hr_max,hr_mean,hr_first,hr_last
0,10000032,29079034,39553978,2180-07-23 14:00:00,2180-07-23 23:50:47,F,52,0,10,91.0,105.0,96.500000,91.0,94.0
1,10000690,25860671,37081114,2150-11-02 19:37:00,2150-11-06 17:03:17,F,86,0,25,62.0,102.0,79.800000,79.0,80.0
2,10000980,26913865,39765666,2189-06-27 08:42:00,2189-06-27 20:38:27,F,76,0,11,68.0,80.0,73.636364,77.0,69.0
3,10001217,24597018,37067082,2157-11-20 19:18:02,2157-11-21 22:08:00,F,55,0,25,78.0,106.0,93.200000,86.0,101.0
4,10001725,25563031,31205490,2110-04-11 15:52:22,2110-04-12 23:59:56,F,46,0,25,55.0,91.0,79.480000,55.0,84.0


In [13]:
# =========================
# Cell 2
# Validation + overwrite-safe save
# =========================

import os

# Basic validation
print("Heart rate summary:")
display(hr_df[['hr_n', 'hr_min', 'hr_max', 'hr_mean', 'hr_first', 'hr_last']].describe())

print("\nRows with no heart rate events:", (hr_df["hr_n"] == 0).sum())

# Overwrite-safe save
hr_path = os.path.join(BASE_PATH, "data", "mimiciv", "mimiciv_heart_rate_24h.csv")
hr_df.to_csv(hr_path, index=False)

print(f"\nSaved to: {hr_path}")

Heart rate summary:


,hr_n,hr_min,hr_max,hr_mean,hr_first,hr_last
count,65366.0,65281.000000,65281.000000,65281.000000,65281.000000,65281.000000
mean,25.93437,69.842086,102.278354,84.092947,87.401088,83.305702
std,9.725564,15.088665,20.525387,15.837979,19.906301,17.719682
min,0.0,30.000000,35.000000,30.769231,30.000000,30.000000
25%,23.0,60.000000,88.000000,72.968750,74.000000,71.000000
50%,25.0,68.000000,100.000000,82.538462,85.000000,82.000000
75%,28.0,79.000000,114.000000,93.893617,99.000000,94.000000
max,1031.0,167.000000,220.000000,174.740741,204.000000,204.000000



Rows with no heart rate events: 85

Saved to: /content/drive/MyDrive/Plan A/data/mimiciv/mimiciv_heart_rate_24h.csv


In [14]:
hr_df[['hr_min','hr_max','hr_mean']].describe()

,hr_min,hr_max,hr_mean
count,65281.000000,65281.000000,65281.000000
mean,69.842086,102.278354,84.092947
std,15.088665,20.525387,15.837979
min,30.000000,35.000000,30.769231
25%,60.000000,88.000000,72.968750
50%,68.000000,100.000000,82.538462
75%,79.000000,114.000000,93.893617
max,167.000000,220.000000,174.740741


In [18]:
# =========================
# Cell 3
# MIMIC-IV: ALL vitals extraction (first 24h)
# Unified, publication-grade, overwrite-safe
# =========================

from google.cloud import bigquery
from google.colab import auth
import os

auth.authenticate_user()
client = bigquery.Client(project="mimic-rs")

BASE_PATH = "/content/drive/MyDrive/Plan A"
mimiciv_dir = os.path.join(BASE_PATH, "data", "mimiciv")

query = """
WITH first_icu AS (
  SELECT
    icu.subject_id,
    icu.hadm_id,
    icu.stay_id,
    icu.intime,
    icu.outtime,
    ROW_NUMBER() OVER (PARTITION BY icu.subject_id ORDER BY icu.intime) AS rn
  FROM `physionet-data.mimiciv_3_1_icu.icustays` icu
),

cohort AS (
  SELECT
    f.subject_id,
    f.hadm_id,
    f.stay_id,
    f.intime,
    f.outtime,
    pat.gender,
    EXTRACT(YEAR FROM f.intime) - pat.anchor_year + pat.anchor_age AS age,
    adm.hospital_expire_flag
  FROM first_icu f
  JOIN `physionet-data.mimiciv_3_1_hosp.patients` pat
    ON f.subject_id = pat.subject_id
  JOIN `physionet-data.mimiciv_3_1_hosp.admissions` adm
    ON f.hadm_id = adm.hadm_id
  WHERE f.rn = 1
    AND (EXTRACT(YEAR FROM f.intime) - pat.anchor_year + pat.anchor_age) >= 18
),

events AS (
  SELECT
    c.stay_id,
    ce.charttime,
    ce.itemid,
    SAFE_CAST(ce.valuenum AS FLOAT64) AS val
  FROM cohort c
  JOIN `physionet-data.mimiciv_3_1_icu.chartevents` ce
    ON ce.stay_id = c.stay_id
   AND ce.charttime >= c.intime
   AND ce.charttime < TIMESTAMP_ADD(c.intime, INTERVAL 24 HOUR)
  WHERE ce.valuenum IS NOT NULL
),

filtered AS (
  SELECT
    stay_id,
    charttime,

    -- Heart Rate
    CASE WHEN itemid = 220045 AND val BETWEEN 30 AND 220 THEN val END AS hr,

    -- SBP
    CASE WHEN itemid = 220179 AND val BETWEEN 50 AND 250 THEN val END AS sbp,

    -- DBP
    CASE WHEN itemid = 220180 AND val BETWEEN 20 AND 150 THEN val END AS dbp,

    -- MAP
    CASE WHEN itemid = 220181 AND val BETWEEN 30 AND 200 THEN val END AS map,

    -- Respiratory Rate
    CASE WHEN itemid = 220210 AND val BETWEEN 5 AND 60 THEN val END AS rr,

    -- SpO2
    CASE WHEN itemid = 220277 AND val BETWEEN 50 AND 100 THEN val END AS spo2,

  FROM events
)

SELECT
  c.subject_id,
  c.hadm_id,
  c.stay_id,
  c.intime,
  c.outtime,
  c.gender,
  c.age,
  c.hospital_expire_flag,

  -- HR
  COUNT(hr) AS hr_n, MIN(hr) AS hr_min, MAX(hr) AS hr_max, AVG(hr) AS hr_mean,

  -- SBP
  COUNT(sbp) AS sbp_n, MIN(sbp) AS sbp_min, MAX(sbp) AS sbp_max, AVG(sbp) AS sbp_mean,

  -- DBP
  COUNT(dbp) AS dbp_n, MIN(dbp) AS dbp_min, MAX(dbp) AS dbp_max, AVG(dbp) AS dbp_mean,

  -- MAP
  COUNT(map) AS map_n, MIN(map) AS map_min, MAX(map) AS map_max, AVG(map) AS map_mean,

  -- RR
  COUNT(rr) AS rr_n, MIN(rr) AS rr_min, MAX(rr) AS rr_max, AVG(rr) AS rr_mean,

  -- SpO2
  COUNT(spo2) AS spo2_n, MIN(spo2) AS spo2_min, MAX(spo2) AS spo2_max, AVG(spo2) AS spo2_mean,



FROM cohort c
LEFT JOIN filtered f
  ON c.stay_id = f.stay_id
GROUP BY
  c.subject_id, c.hadm_id, c.stay_id,
  c.intime, c.outtime, c.gender, c.age, c.hospital_expire_flag
ORDER BY c.subject_id
"""

vitals_df = client.query(query).to_dataframe()

print("Shape:", vitals_df.shape)
display(vitals_df.head())

Shape: (65366, 32)


,subject_id,hadm_id,stay_id,intime,outtime,gender,age,hospital_expire_flag,hr_n,hr_min,...,map_max,map_mean,rr_n,rr_min,rr_max,rr_mean,spo2_n,spo2_min,spo2_max,spo2_mean
0,10000032,29079034,39553978,2180-07-23 14:00:00,2180-07-23 23:50:47,F,52,0,10,91.0,...,67.0,62.300000,10,16.0,24.0,20.700000,10,94.0,99.0,96.300000
1,10000690,25860671,37081114,2150-11-02 19:37:00,2150-11-06 17:03:17,F,86,0,25,62.0,...,82.0,65.166667,25,15.0,35.0,21.840000,24,83.0,100.0,94.916667
2,10000980,26913865,39765666,2189-06-27 08:42:00,2189-06-27 20:38:27,F,76,0,11,68.0,...,135.0,97.545455,11,14.0,25.0,20.545455,11,96.0,100.0,98.909091
3,10001217,24597018,37067082,2157-11-20 19:18:02,2157-11-21 22:08:00,F,55,0,25,78.0,...,130.0,93.041667,25,13.0,27.0,21.320000,25,92.0,99.0,96.080000
4,10001725,25563031,31205490,2110-04-11 15:52:22,2110-04-12 23:59:56,F,46,0,25,55.0,...,91.0,68.360000,25,13.0,21.0,17.360000,24,94.0,100.0,98.125000


In [20]:
# =========================
# Cell 4
# Validation + Save
# =========================

import os

print("Quick sanity check:")

display(vitals_df[['hr_mean','sbp_mean','dbp_mean','map_mean','rr_mean','spo2_mean']].describe())

print("\nMissing counts per vital:")
print({
    'hr_missing': (vitals_df['hr_n'] == 0).sum(),
    'sbp_missing': (vitals_df['sbp_n'] == 0).sum(),
    'dbp_missing': (vitals_df['dbp_n'] == 0).sum(),
    'map_missing': (vitals_df['map_n'] == 0).sum(),
    'rr_missing': (vitals_df['rr_n'] == 0).sum(),
    'spo2_missing': (vitals_df['spo2_n'] == 0).sum(),

})

# Save
save_path = os.path.join(BASE_PATH, "data/mimiciv/mimiciv_vitals_24h.csv")
vitals_df.to_csv(save_path, index=False)

print(f"\nSaved to: {save_path}")

Quick sanity check:


,hr_mean,sbp_mean,dbp_mean,map_mean,rr_mean,spo2_mean
count,65281.000000,59236.000000,59234.000000,59202.000000,65126.000000,65201.000000
mean,84.092947,118.526717,65.848261,79.175958,19.037631,96.779072
std,15.837979,17.556170,11.938181,12.349945,3.782579,2.269598
min,30.769231,52.000000,20.000000,30.000000,6.000000,52.000000
25%,72.968750,105.800000,57.600000,70.500000,16.393128,95.625000
50%,82.538462,116.576923,64.928571,78.000000,18.387097,97.000000
75%,93.893617,129.541667,73.200000,86.791667,21.000000,98.304348
max,174.740741,247.000000,138.000000,162.000000,44.333333,100.000000



Missing counts per vital:
{'hr_missing': np.int64(85), 'sbp_missing': np.int64(6130), 'dbp_missing': np.int64(6132), 'map_missing': np.int64(6164), 'rr_missing': np.int64(240), 'spo2_missing': np.int64(165)}

Saved to: /content/drive/MyDrive/Plan A/data/mimiciv/mimiciv_vitals_24h.csv


In [21]:
# =========================
# Cell 5
# MIMIC-IV: Creatinine (first 24h)
# =========================

CREATININE_ITEMIDS = [50912]

query = f"""
WITH first_icu AS (
  SELECT
    icu.subject_id,
    icu.hadm_id,
    icu.stay_id,
    icu.intime,
    ROW_NUMBER() OVER (PARTITION BY icu.subject_id ORDER BY icu.intime) AS rn
  FROM `physionet-data.mimiciv_3_1_icu.icustays` icu
),

cohort AS (
  SELECT
    f.subject_id,
    f.hadm_id,
    f.stay_id,
    f.intime,
    EXTRACT(YEAR FROM f.intime) - pat.anchor_year + pat.anchor_age AS age
  FROM first_icu f
  JOIN `physionet-data.mimiciv_3_1_hosp.patients` pat
    ON f.subject_id = pat.subject_id
  WHERE f.rn = 1
    AND (EXTRACT(YEAR FROM f.intime) - pat.anchor_year + pat.anchor_age) >= 18
),

lab_events AS (
  SELECT
    c.stay_id,
    le.charttime,
    SAFE_CAST(le.valuenum AS FLOAT64) AS creatinine
  FROM cohort c
  JOIN `physionet-data.mimiciv_3_1_hosp.labevents` le
    ON c.hadm_id = le.hadm_id
   AND le.charttime >= c.intime
   AND le.charttime < TIMESTAMP_ADD(c.intime, INTERVAL 24 HOUR)
   AND le.itemid IN ({','.join(map(str, CREATININE_ITEMIDS))})
  WHERE le.valuenum IS NOT NULL
    AND SAFE_CAST(le.valuenum AS FLOAT64) BETWEEN 0.1 AND 20
)

SELECT
  c.subject_id,
  c.hadm_id,
  c.stay_id,
  COUNT(l.creatinine) AS creat_n,
  MIN(l.creatinine) AS creat_min,
  MAX(l.creatinine) AS creat_max,
  AVG(l.creatinine) AS creat_mean
FROM cohort c
LEFT JOIN lab_events l
  ON c.stay_id = l.stay_id
GROUP BY c.subject_id, c.hadm_id, c.stay_id
ORDER BY c.subject_id
"""

creat_df = client.query(query).to_dataframe()

print("Shape:", creat_df.shape)
display(creat_df.head())

Shape: (65366, 7)


,subject_id,hadm_id,stay_id,creat_n,creat_min,creat_max,creat_mean
0,10000032,29079034,39553978,2,0.4,0.5,0.45
1,10000690,25860671,37081114,1,0.9,0.9,0.90
2,10000980,26913865,39765666,1,2.2,2.2,2.20
3,10001217,24597018,37067082,1,0.4,0.4,0.40
4,10001725,25563031,31205490,2,0.8,0.8,0.80


In [22]:
# =========================
# Cell 6
# Validation + Save
# =========================

print("Creatinine summary:")
display(creat_df[['creat_min','creat_max','creat_mean']].describe())

print("\nMissing:", (creat_df['creat_n'] == 0).sum())

save_path = os.path.join(BASE_PATH, "data/mimiciv/mimiciv_creatinine_24h.csv")
creat_df.to_csv(save_path, index=False)

print(f"Saved to: {save_path}")

Creatinine summary:


,creat_min,creat_max,creat_mean
count,63286.000000,63286.000000,63286.000000
mean,1.229079,1.393771,1.311116
std,1.266334,1.479459,1.361646
min,0.100000,0.100000,0.100000
25%,0.700000,0.700000,0.700000
50%,0.900000,1.000000,0.900000
75%,1.200000,1.400000,1.300000
max,19.900000,20.000000,19.900000



Missing: 2080
Saved to: /content/drive/MyDrive/Plan A/data/mimiciv/mimiciv_creatinine_24h.csv


In [ ]:
# =========================
# Cell 7
# MIMIC-IV: Lactate (first 24h)
# =========================

LACTATE_ITEMIDS = [50813]

query = f"""
WITH first_icu AS (
  SELECT
    icu.subject_id,
    icu.hadm_id,
    icu.stay_id,
    icu.intime,
    ROW_NUMBER() OVER (PARTITION BY icu.subject_id ORDER BY icu.intime) AS rn
  FROM `physionet-data.mimiciv_3_1_icu.icustays` icu
),

cohort AS (
  SELECT
    f.subject_id,
    f.hadm_id,
    f.stay_id,
    f.intime,
    EXTRACT(YEAR FROM f.intime) - pat.anchor_year + pat.anchor_age AS age
  FROM first_icu f
  JOIN `physionet-data.mimiciv_3_1_hosp.patients` pat
    ON f.subject_id = pat.subject_id
  WHERE f.rn = 1
    AND (EXTRACT(YEAR FROM f.intime) - pat.anchor_year + pat.anchor_age) >= 18
),

lab_events AS (
  SELECT
    c.stay_id,
    le.charttime,
    SAFE_CAST(le.valuenum AS FLOAT64) AS lactate
  FROM cohort c
  JOIN `physionet-data.mimiciv_3_1_hosp.labevents` le
    ON c.hadm_id = le.hadm_id
   AND le.charttime >= c.intime
   AND le.charttime < TIMESTAMP_ADD(c.intime, INTERVAL 24 HOUR)
   AND le.itemid IN ({','.join(map(str, LACTATE_ITEMIDS))})
  WHERE le.valuenum IS NOT NULL
    AND SAFE_CAST(le.valuenum AS FLOAT64) BETWEEN 0.2 AND 20
)

SELECT
  c.subject_id,
  c.hadm_id,
  c.stay_id,
  COUNT(l.lactate) AS lact_n,
  MIN(l.lactate) AS lact_min,
  MAX(l.lactate) AS lact_max,
  AVG(l.lactate) AS lact_mean
FROM cohort c
LEFT JOIN lab_events l
  ON c.stay_id = l.stay_id
GROUP BY c.subject_id, c.hadm_id, c.stay_id
ORDER BY c.subject_id
"""

lact_df = client.query(query).to_dataframe()

print("Shape:", lact_df.shape)
display(lact_df.head())